# AML Stats — KPIs de negócio sobre as previsões

Notebook batch que fecha o ciclo: lê as previsões geradas pelos consumers (Kafka e/ou file-based),
deriva uma tabela de dimensão `dim_banks` a partir das transações originais e produz indicadores
de negócio agregados.

Também regista tabelas externas no metastore Hive para permitir consultas SQL ad-hoc.

**Pré-requisitos:**
1. `aml_trainer.ipynb` executado (modelo persistido).
2. Pelo menos um dos consumers a correr (ou já tendo corrido) para produzir `predictions_*/`.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField,
    TimestampType, IntegerType, StringType, DoubleType,
)

HDFS_BASE       = "hdfs://10.84.129.52:9000/aulas/fabricio_moreira/project"
SOURCE_MEDIUM   = f"{HDFS_BASE}/dataset/LI-Medium_Trans.csv"
PRED_KAFKA      = f"{HDFS_BASE}/stream/predictions_kafka"
PRED_FILE       = f"{HDFS_BASE}/stream/predictions_file"
DIM_BANKS_PATH  = f"{HDFS_BASE}/dim/banks"

spark = SparkSession.builder \
    .appName("AML Stats") \
    .enableHiveSupport() \
    .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

## 1) Tabela de dimensão `dim_banks`

Derivada das transações: cada `bank_id` distinto recebe nome sintético, país e tier.
Os atributos (`country`, `tier`) são fabricados a partir do ID para fins de demonstração — num cenário
real esta tabela viria do core bancário ou de uma fonte regulatória.

In [ ]:
aml_schema = StructType([
    StructField("Timestamp", TimestampType(), True),
    StructField("From Bank", IntegerType(), True),
    StructField("Account2", StringType(), True),
    StructField("To Bank", IntegerType(), True),
    StructField("Account4", StringType(), True),
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True),
])

df_tx = spark.read \
    .option("header", True) \
    .option("timestampFormat", "yyyy/MM/dd HH:mm") \
    .schema(aml_schema) \
    .csv(SOURCE_MEDIUM)

banks = (
    df_tx.select(F.col("From Bank").alias("bank_id"))
         .union(df_tx.select(F.col("To Bank").alias("bank_id")))
         .distinct()
)

dim_banks = (
    banks
      .withColumn("bank_name", F.concat(F.lit("Bank_"), F.col("bank_id")))
      .withColumn(
          "country",
          F.when(F.col("bank_id") < 1000, "PT")
           .when(F.col("bank_id") < 5000, "ES")
           .when(F.col("bank_id") < 20000, "FR")
           .otherwise("OTHER"),
      )
      .withColumn(
          "tier",
          F.when(F.col("bank_id") < 100, "tier1")
           .when(F.col("bank_id") < 1000, "tier2")
           .otherwise("tier3"),
      )
)

dim_banks.write.mode("overwrite").parquet(DIM_BANKS_PATH)
dim_banks = spark.read.parquet(DIM_BANKS_PATH).cache()
print(f"dim_banks: {dim_banks.count():,} bancos distintos")
dim_banks.show(5, truncate=False)

## 2) Carregar previsões

Une previsões de ambos os sinks (Kafka + file-based). Se só um existir, usa esse.

In [ ]:
def safe_read(path):
    try:
        d = spark.read.parquet(path)
        return d if d.count() > 0 else None
    except Exception as e:
        print(f"sem dados em {path}: {type(e).__name__}")
        return None

p_kafka = safe_read(PRED_KAFKA)
p_file  = safe_read(PRED_FILE)

# Normaliza as duas fontes (kafka tem kafka_ts, file tem ingest_ts)
common_cols = [
    "Timestamp", "From Bank", "To Bank", "Account2", "Account4",
    "Amount Paid", "Receiving Currency", "Payment Format",
    "truth", "prediction", "prob_fraud",
]

frames = []
if p_kafka is not None:
    frames.append(p_kafka.withColumn("source", F.lit("kafka")).select(*common_cols, "source"))
if p_file is not None:
    frames.append(p_file.withColumn("source", F.lit("file")).select(*common_cols, "source"))

if not frames:
    raise RuntimeError("Nenhuma fonte de previsões disponível. Corre o producer + consumer primeiro.")

preds = frames[0] if len(frames) == 1 else frames[0].unionByName(frames[1])
preds.cache()
print(f"Previsões carregadas: {preds.count():,}")
preds.groupBy("source").count().show()

## 3) KPIs globais

In [ ]:
s = preds.agg(
    F.count("*").alias("total_tx"),
    F.sum("prediction").alias("n_suspeita"),
    F.sum(F.col("Amount Paid")).alias("valor_total"),
    F.sum(F.when(F.col("prediction") == 1, F.col("Amount Paid"))).alias("valor_suspeita"),
    F.sum("truth").alias("n_truth_fraude"),
    F.sum(F.when(F.col("truth") == 1, F.col("Amount Paid"))).alias("valor_truth_fraude"),
).collect()[0]

n_total   = s["total_tx"]
n_susp    = s["n_suspeita"] or 0
n_ok      = n_total - n_susp
val_total = s["valor_total"] or 0
val_susp  = s["valor_suspeita"] or 0

print("=" * 60)
print("KPIs Globais")
print("=" * 60)
print(f"Total de transações analisadas:  {n_total:>14,}")
print(f"  Fidedignas (prediction=0):     {n_ok:>14,}  ({n_ok/n_total:>6.2%})")
print(f"  Suspeitas  (prediction=1):     {n_susp:>14,}  ({n_susp/n_total:>6.2%})")
print()
print(f"Valor total transacionado:       {val_total:>18,.2f}")
print(f"Valor em transações suspeitas:   {val_susp:>18,.2f}  ({val_susp/val_total:>6.2%})")
print()
print(f"[ref. ground truth] fraude real: {s['n_truth_fraude']:>14,}  ({s['valor_truth_fraude']:>14,.2f} EUR equiv.)")

## 4) Fraude por instituição bancária (`From Bank`)

Join com `dim_banks` para nome legível.

In [ ]:
fraud_by_bank = (
    preds.filter("prediction = 1")
         .join(dim_banks, preds["From Bank"] == dim_banks["bank_id"])
         .groupBy("bank_name", "country", "tier")
         .agg(
             F.count("*").alias("n_tx_suspeitas"),
             F.sum(F.col("Amount Paid")).alias("valor_suspeito"),
             F.avg("prob_fraud").alias("prob_media"),
         )
         .orderBy(F.desc("valor_suspeito"))
)

print("Top 15 bancos por valor de transações suspeitas (lado 'From'):")
fraud_by_bank.show(15, truncate=False)

In [ ]:
fraud_by_country = (
    preds.join(dim_banks, preds["From Bank"] == dim_banks["bank_id"])
         .groupBy("country")
         .agg(
             F.count("*").alias("n_tx"),
             F.sum("prediction").alias("n_suspeitas"),
             F.sum(F.when(F.col("prediction") == 1, F.col("Amount Paid"))).alias("valor_suspeito"),
         )
         .withColumn("taxa_suspeita", F.col("n_suspeitas") / F.col("n_tx"))
         .orderBy(F.desc("valor_suspeito"))
)
print("Fraude por país (dim_banks sintético):")
fraud_by_country.show()

## 5) Fraude por conta — top contas suspeitas

In [ ]:
fraud_by_account = (
    preds.filter("prediction = 1")
         .groupBy(F.col("Account2").alias("account_from"), F.col("From Bank").alias("bank_id"))
         .agg(
             F.count("*").alias("n_suspeitas"),
             F.sum(F.col("Amount Paid")).alias("valor_suspeito"),
             F.avg("prob_fraud").alias("prob_media"),
         )
         .join(dim_banks, "bank_id")
         .select("account_from", "bank_name", "country", "n_suspeitas", "valor_suspeito", "prob_media")
         .orderBy(F.desc("valor_suspeito"))
)

print("Top 20 contas com maior valor em transações suspeitas:")
fraud_by_account.show(20, truncate=False)

## 6) Tabelas externas Hive

Cria tabelas externas sobre os parquets para permitir queries SQL ad-hoc no Hive/Spark SQL.
Cumpre o requisito do enunciado de "Spark, MapReduce, **Hive**, ...".

Se o cluster não tiver metastore Hive configurado, este bloco falha sem afectar o resto do notebook.

In [ ]:
try:
    spark.sql("CREATE DATABASE IF NOT EXISTS aml_fabricio")
    spark.sql("USE aml_fabricio")

    spark.sql("DROP TABLE IF EXISTS dim_banks")
    spark.sql(f"""
        CREATE EXTERNAL TABLE dim_banks (
          bank_id   INT,
          bank_name STRING,
          country   STRING,
          tier      STRING
        )
        STORED AS PARQUET
        LOCATION '{DIM_BANKS_PATH}'
    """)

    for tbl, loc, ts_col in [
        ("predictions_kafka", PRED_KAFKA, "kafka_ts"),
        ("predictions_file",  PRED_FILE,  "ingest_ts"),
    ]:
        spark.sql(f"DROP TABLE IF EXISTS {tbl}")
        spark.sql(f"""
            CREATE EXTERNAL TABLE {tbl} (
              {ts_col}             TIMESTAMP,
              `Timestamp`          TIMESTAMP,
              `From Bank`          INT,
              `To Bank`            INT,
              Account2             STRING,
              Account4             STRING,
              `Amount Paid`        DOUBLE,
              `Receiving Currency` STRING,
              `Payment Format`     STRING,
              truth                INT,
              prediction           INT,
              prob_fraud           DOUBLE,
              status               STRING
            )
            PARTITIONED BY (ingest_hour STRING)
            STORED AS PARQUET
            LOCATION '{loc}'
        """)
        spark.sql(f"MSCK REPAIR TABLE {tbl}")

    print("Tabelas Hive criadas:")
    spark.sql("SHOW TABLES IN aml_fabricio").show()
except Exception as e:
    print(f"Hive metastore indisponível ou erro: {e}")
    print("As secções seguintes deste notebook ainda funcionam — só não terás SQL ad-hoc via Hive.")

## 7) Exemplo de query SQL via Hive

Demonstra que os dados já estão consultáveis com SQL puro pelo metastore.

In [ ]:
try:
    spark.sql("""
        SELECT b.country,
               b.tier,
               COUNT(*)               AS n_suspeitas,
               ROUND(SUM(p.`Amount Paid`), 2) AS valor_suspeito
        FROM aml_fabricio.predictions_kafka p
        JOIN aml_fabricio.dim_banks        b ON p.`From Bank` = b.bank_id
        WHERE p.prediction = 1
        GROUP BY b.country, b.tier
        ORDER BY valor_suspeito DESC
    """).show()
except Exception as e:
    print(f"Query falhou (Hive indisponível?): {e}")